In [1]:

import re

samples = [
    "AI, search!",
    "I can't log in.",
    "help@example.com",
    "CS4944 version 2.1",
    "",
    "یہ ایک مثال ہے"
]

for text in samples:
    ws = text.split()
    words = re.findall(r"\b\w+\b", text)

    print("Original:", repr(text))
    print("Whitespace:", ws)
    print("Count:", len(ws))
    print("Word-like:", words)
    print("Count:", len(words))
    print("------------------")


Original: 'AI, search!'
Whitespace: ['AI,', 'search!']
Count: 2
Word-like: ['AI', 'search']
Count: 2
------------------
Original: "I can't log in."
Whitespace: ['I', "can't", 'log', 'in.']
Count: 4
Word-like: ['I', 'can', 't', 'log', 'in']
Count: 5
------------------
Original: 'help@example.com'
Whitespace: ['help@example.com']
Count: 1
Word-like: ['help', 'example', 'com']
Count: 3
------------------
Original: 'CS4944 version 2.1'
Whitespace: ['CS4944', 'version', '2.1']
Count: 3
Word-like: ['CS4944', 'version', '2', '1']
Count: 4
------------------
Original: ''
Whitespace: []
Count: 0
Word-like: []
Count: 0
------------------
Original: 'یہ ایک مثال ہے'
Whitespace: ['یہ', 'ایک', 'مثال', 'ہے']
Count: 4
Word-like: ['یہ', 'ایک', 'مثال', 'ہے']
Count: 4
------------------


In [2]:

!pip install sentence-transformers


In [3]:

from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu"
)

for text in samples[:2]:
    print("Original:", text)
    print("Tokens:", model.tokenizer.tokenize(text))
    print("------------------")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Original: AI, search!
Tokens: ['ai', ',', 'search', '!']
------------------
Original: I can't log in.
Tokens: ['i', 'can', "'", 't', 'log', 'in', '.']
------------------


In [4]:

!pip install scikit-learn


In [7]:

%%writefile lab6_search.py

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer


class Retriever:

    def __init__(self, records, backend="tfidf"):

        if not records:
            raise ValueError("Provide nonempty records")

        self.records = records
        self.backend = backend

        texts = [r["text"] for r in records]

        if backend == "tfidf":

            self.vectorizer = TfidfVectorizer()
            self.matrix = self.vectorizer.fit_transform(texts)

        else:
            raise ValueError("Use tfidf")

    def search(self, query, k=3):

        if not query.strip():
            return []

        q = self.vectorizer.transform([query])

        if q.nnz == 0:
            return []

        scores = (q @ self.matrix.T).toarray()[0]

        ranked = sorted(
            range(len(self.records)),
            key=lambda i: (-float(scores[i]), i)
        )

        return [
            (self.records[i], float(scores[i]))
            for i in ranked[:k]
        ]


Overwriting lab6_search.py


In [8]:

%%writefile guided_chatbot.py

from lab6_search import Retriever

FAQ = [
    {
        "id": "faq:password",
        "question": "How do I reset my password?",
        "answer": "Use the recovery link on the login page.",
        "source": "it_help.txt"
    },
    {
        "id": "faq:submit",
        "question": "Where do I submit my assignment?",
        "answer": "Use the assignment upload page in the learning portal.",
        "source": "assessment.txt"
    },
    {
        "id": "faq:late",
        "question": "Can illness justify a late submission?",
        "answer": "Request an extension with medical evidence before the deadline.",
        "source": "handbook.txt"
    },
    {
        "id": "faq:wifi",
        "question": "How do I connect to campus Wi-Fi?",
        "answer": "Use your university username and password to connect.",
        "source": "student_guide.txt"
    },
    {
        "id": "faq:lms",
        "question": "How do I create a group in LMS?",
        "answer": "Open the course group section and use the group creation option if available.",
        "source": "lms_help.txt"
    }
]

def build_bot(backend="tfidf"):
    records = [
        dict(r, text=r["question"] + " " + r["answer"])
        for r in FAQ
    ]
    return Retriever(records, backend)

def reply(engine, query):
    if not query.strip():
        return "Please enter a question."

    for r in FAQ:
        if query.strip().casefold() == r["question"].casefold():
            return r["answer"] + " [source: " + r["source"] + "]"

    hits = engine.search(query)

    if not hits:
        return "No matching FAQ. Please contact the help desk."

    lines = [
        "Possible matches; choose or rephrase your question:"
    ]

    for r, score in hits:
        lines.append(r["question"] + f" (similarity {score:.3f})")

    return "\n".join(lines)


Writing guided_chatbot.py


In [9]:

from guided_chatbot import build_bot, reply

bot = build_bot()

# Test 1: Exact question
print("Test 1:")
print(reply(bot, "How do I reset my password?"))

# Test 2: Similar question
print("\nTest 2:")
print(reply(bot, "I forgot my password"))

# Test 3: Empty input
print("\nTest 3:")
print(reply(bot, ""))

# Test 4: Unsupported question
print("\nTest 4:")
print(reply(bot, "What is volcano geology?"))


Test 1:
Use the recovery link on the login page. [source: it_help.txt]

Test 2:
Possible matches; choose or rephrase your question:
How do I reset my password? (similarity 0.374)
Where do I submit my assignment? (similarity 0.159)
How do I connect to campus Wi-Fi? (similarity 0.141)

Test 3:
Please enter a question.

Test 4:
No matching FAQ. Please contact the help desk.


In [10]:

%%writefile task2_chunks.py

import re

def word_spans(text):
    return [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]

def windows(text, size=40, overlap=0, offset=0):
    if size <= 0 or not 0 <= overlap < size:
        raise ValueError("Require size > 0 and 0 <= overlap < size")

    spans = word_spans(text)
    result = []

    for i in range(0, len(spans), size - overlap):
        j = min(i + size, len(spans))

        result.append((
            offset + spans[i][0],
            offset + spans[j - 1][1]
        ))

        if j == len(spans):
            break

    return result

def sentence_chunks(text, budget=40):
    if budget <= 0:
        raise ValueError("Budget must be positive")

    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    chunks = []
    current = []

    for sentence in sentences:
        words = sentence.split()

        if not words:
            continue

        if len(words) > budget:
            if current:
                chunks.append(" ".join(current))
                current = []

            for i in range(0, len(words), budget):
                chunks.append(" ".join(words[i:i+budget]))

        elif len(current) + len(words) <= budget:
            current.extend(words)

        else:
            chunks.append(" ".join(current))
            current = words

    if current:
        chunks.append(" ".join(current))

    return chunks


Writing task2_chunks.py


In [12]:

documents = {
    "doc1": """Students must submit assignments before the deadline.
Late submissions require instructor approval.
Students can request an extension when illness affects their work.
The request should include supporting medical evidence.
Assignments must be uploaded through the LMS.
Students should check the uploaded file before submission.
The instructor may reject incomplete submissions.
Students are responsible for keeping a copy of their work.
Students should read the assignment instructions carefully.
The deadline is mentioned on the learning portal.
Students must ensure that their files are readable.
If technical problems occur, students should contact IT support.
An extension should be requested before the deadline.
However, an extension is not automatically granted.
The instructor reviews each request individually.
Students should provide accurate information.
Late work may receive a penalty.
The penalty depends on course policy.
Students should avoid waiting until the last minute.
They should confirm successful submission.
The LMS may display a submission receipt.
Students should save that receipt for reference.
""",

    "doc2": """Campus Wi-Fi is available to registered students.
Students must use their university login credentials.
The network should only be used for educational purposes.
Sharing account passwords is not allowed.
If the connection fails, students should contact IT support.
Students must not attempt to access restricted systems.
The university may temporarily disable accounts for security reasons.
Students should connect using the official network name.
They should enter their username and password carefully.
A successful connection allows access to online learning resources.
Students can use the network in approved campus areas.
They should avoid sharing personal information on public websites.
The university recommends keeping devices updated.
Students should report suspicious network activity.
Technical support may ask for the device type.
They may also ask when the problem started.
Students should not change network settings without guidance.
If the password expires, students should reset it.
The IT department can help with login problems.
Students should follow university technology policies.
Network access may be limited during maintenance.
Students should check official announcements for updates.
""",

    "doc3": """Students can create groups through the learning portal.
Each group should include the required number of members.
The group leader is responsible for coordinating activities.
Students should confirm their group before submitting work.
If a student cannot join a group, they should contact the instructor.
Group changes may require instructor approval.
Students must follow the assignment instructions.
The group should select a leader before starting.
Members should divide the assigned tasks fairly.
Each student should understand their responsibilities.
The group leader should communicate important deadlines.
Students should share relevant information with teammates.
They should review the completed work together.
The final submission must follow the course format.
Members should check the document for missing sections.
The instructor may ask groups to explain their work.
Students should keep copies of their contributions.
If disagreements occur, members should discuss them respectfully.
The instructor can help resolve serious group problems.
Students should not submit another group's work.
Every member should participate in the assignment.
The group should confirm the submission before the deadline.
"""
}

for name, text in documents.items():
    print(name, "words:", len(text.split()))


doc1 words: 157
doc2 words: 170
doc3 words: 170


In [13]:

from task2_chunks import windows

for name, text in documents.items():
    spans = windows(text, size=40, overlap=0)

    print("\n", name)
    print("Total chunks:", len(spans))

    for i, (start, end) in enumerate(spans):
        chunk = text[start:end]
        print("Chunk", i + 1)
        print("Start:", start, "End:", end)
        print("Words:", len(chunk.split()))
        print(chunk)
        print()



 doc1
Total chunks: 4
Chunk 1
Start: 0 End: 294
Words: 40
Students must submit assignments before the deadline.
Late submissions require instructor approval.
Students can request an extension when illness affects their work.
The request should include supporting medical evidence.
Assignments must be uploaded through the LMS.
Students should check the

Chunk 2
Start: 295 End: 578
Words: 40
uploaded file before submission.
The instructor may reject incomplete submissions.
Students are responsible for keeping a copy of their work.
Students should read the assignment instructions carefully.
The deadline is mentioned on the learning portal.
Students must ensure that their

Chunk 3
Start: 579 End: 875
Words: 40
files are readable.
If technical problems occur, students should contact IT support.
An extension should be requested before the deadline.
However, an extension is not automatically granted.
The instructor reviews each request individually.
Students should provide accurate informatio

In [14]:

from task2_chunks import windows

for name, text in documents.items():
    spans = windows(text, size=40, overlap=10)

    print("\n", name)
    print("Total chunks:", len(spans))

    for i, (start, end) in enumerate(spans):
        chunk = text[start:end]

        print("Chunk", i + 1)
        print("Start:", start, "End:", end)
        print("Words:", len(chunk.split()))
        print(chunk)
        print()



 doc1
Total chunks: 5
Chunk 1
Start: 0 End: 294
Words: 40
Students must submit assignments before the deadline.
Late submissions require instructor approval.
Students can request an extension when illness affects their work.
The request should include supporting medical evidence.
Assignments must be uploaded through the LMS.
Students should check the

Chunk 2
Start: 235 End: 512
Words: 40
must be uploaded through the LMS.
Students should check the uploaded file before submission.
The instructor may reject incomplete submissions.
Students are responsible for keeping a copy of their work.
Students should read the assignment instructions carefully.
The deadline is

Chunk 3
Start: 437 End: 717
Words: 40
Students should read the assignment instructions carefully.
The deadline is mentioned on the learning portal.
Students must ensure that their files are readable.
If technical problems occur, students should contact IT support.
An extension should be requested before the deadline.

Chunk 4


In [15]:

%%writefile task2_chunks.py

import re

def word_spans(text):
    return [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]


def windows(text, size=40, overlap=0):
    if size <= 0 or overlap < 0 or overlap >= size:
        raise ValueError("Invalid size or overlap")

    spans = word_spans(text)
    result = []

    for i in range(0, len(spans), size - overlap):
        j = min(i + size, len(spans))
        result.append((spans[i][0], spans[j - 1][1]))

        if j == len(spans):
            break

    return result


def sentence_chunks(text, size=40):
    if size <= 0:
        raise ValueError("Size must be positive")

    result = []
    sentences = list(re.finditer(r".+?(?:[.!?](?=\s|$)|$)", text, re.S))

    current_start = None
    current_end = None
    current_words = 0

    for sentence in sentences:
        start = sentence.start()
        end = sentence.end()
        words = word_spans(sentence.group())

        if not words:
            continue

        count = len(words)

        if count > size:
            if current_start is not None:
                result.append((current_start, current_end))
                current_start = None
                current_words = 0

            for i in range(0, count, size):
                part = words[i:i + size]
                result.append((
                    start + part[0][0],
                    start + part[-1][1]
                ))

        elif current_words + count <= size:
            if current_start is None:
                current_start = start

            current_end = end
            current_words += count

        else:
            result.append((current_start, current_end))
            current_start = start
            current_end = end
            current_words = count

    if current_start is not None:
        result.append((current_start, current_end))

    return result


Overwriting task2_chunks.py


In [17]:
import task2_chunks
import importlib

importlib.reload(task2_chunks)

print(task2_chunks.sentence_chunks.__code__.co_varnames)

('text', 'size', 'result', 'sentences', 'current_start', 'current_end', 'current_words', 'sentence', 'start', 'end', 'words', 'count', 'i', 'part')


In [18]:
from task2_chunks import sentence_chunks

for name, text in documents.items():
    spans = sentence_chunks(text, size=40)

    print("\n", name)
    print("Total chunks:", len(spans))

    for i, (start, end) in enumerate(spans):
        chunk = text[start:end]

        print("Chunk", i + 1)
        print("Start:", start, "End:", end)
        print("Words:", len(chunk.split()))
        print(chunk)
        print()


 doc1
Total chunks: 5
Chunk 1
Start: 0 End: 268
Words: 36
Students must submit assignments before the deadline.
Late submissions require instructor approval.
Students can request an extension when illness affects their work.
The request should include supporting medical evidence.
Assignments must be uploaded through the LMS.

Chunk 2
Start: 268 End: 546
Words: 39

Students should check the uploaded file before submission.
The instructor may reject incomplete submissions.
Students are responsible for keeping a copy of their work.
Students should read the assignment instructions carefully.
The deadline is mentioned on the learning portal.

Chunk 3
Start: 546 End: 819
Words: 38

Students must ensure that their files are readable.
If technical problems occur, students should contact IT support.
An extension should be requested before the deadline.
However, an extension is not automatically granted.
The instructor reviews each request individually.

Chunk 4
Start: 819 End: 1074
Words: 37



In [19]:
from task2_chunks import windows, sentence_chunks

# 1. Empty text
print("Empty:", sentence_chunks(""))

# 2. Text shorter than 40 words
short_text = "Students should follow university rules."
print("Short:", sentence_chunks(short_text))

# 3. Invalid overlap
try:
    windows("This is a test.", size=40, overlap=40)
except ValueError as e:
    print("Invalid overlap:", e)

# 4. Sentence longer than 40 words
long_sentence = " ".join(["Students"] * 50) + "."
print("Long sentence:", sentence_chunks(long_sentence, size=40))

Empty: []
Short: [(0, 40)]
Invalid overlap: Invalid size or overlap
Long sentence: [(0, 359), (360, 450)]


In [21]:
text = "Sir. Rasikh teaches AI. Students attend his class."

spans = sentence_chunks(text, size=40)

for start, end in spans:
    print(repr(text[start:end]))

'Sir. Rasikh teaches AI. Students attend his class.'


In [27]:
%%writefile task2_chunks.py
def make_chunks(docs, strategy):
    result = []

    for doc_id, text in docs.items():
        if strategy == "sentence":
            spans = sentence_chunks(text)
        else:
            spans = windows(text, 40, int(strategy))

        for i, (a, b) in enumerate(spans):
            result.append({
                "id": f"{doc_id}:{i}",
                "doc": doc_id,
                "start": a,
                "end": b,
                "text": text[a:b]
            })

    return result

Overwriting task2_chunks.py


In [28]:
import importlib
import task2_chunks

importlib.reload(task2_chunks)

print("make_chunks" in dir(task2_chunks))

True


In [29]:
from task2_chunks import make_chunks

for strategy in ["0", "10", "sentence"]:
    chunks = make_chunks(documents, strategy)

    print("\nStrategy:", strategy)
    print("Total chunks:", len(chunks))

    for c in chunks[:3]:
        print(c["id"], c["start"], c["end"])
        print(c["text"][:100])

        print(
            "Source check:",
            c["text"] == documents[c["doc"]][c["start"]:c["end"]]
        )


Strategy: 0
Total chunks: 14
doc1:0 0 294
Students must submit assignments before the deadline.
Late submissions require instructor approval.

Source check: True
doc1:1 295 578
uploaded file before submission.
The instructor may reject incomplete submissions.
Students are resp
Source check: True
doc1:2 579 875
files are readable.
If technical problems occur, students should contact IT support.
An extension sh
Source check: True

Strategy: 10
Total chunks: 17
doc1:0 0 294
Students must submit assignments before the deadline.
Late submissions require instructor approval.

Source check: True
doc1:1 235 512
must be uploaded through the LMS.
Students should check the uploaded file before submission.
The ins
Source check: True
doc1:2 437 717
Students should read the assignment instructions carefully.
The deadline is mentioned on the learnin
Source check: True

Strategy: sentence
Total chunks: 15
doc1:0 0 268
Students must submit assignments before the deadline.
Late submissions require inst

In [30]:
for strategy in ["0", "10", "sentence"]:
    chunks = make_chunks(documents, strategy)

    print("\nStrategy:", strategy)

    for c in chunks:
        print(c["id"], "Words:", len(c["text"].split()))


Strategy: 0
doc1:0 Words: 40
doc1:1 Words: 40
doc1:2 Words: 40
doc1:3 Words: 37
doc2:0 Words: 40
doc2:1 Words: 40
doc2:2 Words: 40
doc2:3 Words: 40
doc2:4 Words: 10
doc3:0 Words: 40
doc3:1 Words: 40
doc3:2 Words: 40
doc3:3 Words: 40
doc3:4 Words: 10

Strategy: 10
doc1:0 Words: 40
doc1:1 Words: 40
doc1:2 Words: 40
doc1:3 Words: 40
doc1:4 Words: 37
doc2:0 Words: 40
doc2:1 Words: 40
doc2:2 Words: 40
doc2:3 Words: 40
doc2:4 Words: 40
doc2:5 Words: 20
doc3:0 Words: 40
doc3:1 Words: 40
doc3:2 Words: 40
doc3:3 Words: 40
doc3:4 Words: 40
doc3:5 Words: 20

Strategy: sentence
doc1:0 Words: 36
doc1:1 Words: 39
doc1:2 Words: 38
doc1:3 Words: 37
doc1:4 Words: 7
doc2:0 Words: 38
doc2:1 Words: 33
doc2:2 Words: 39
doc2:3 Words: 40
doc2:4 Words: 20
doc3:0 Words: 33
doc3:1 Words: 39
doc3:2 Words: 35
doc3:3 Words: 40
doc3:4 Words: 23


In [31]:
for c in make_chunks(documents, "sentence"):
    if "rule" in c["text"].lower() or "exception" in c["text"].lower():
        print("\nChunk ID:", c["id"])
        print(c["text"])

In [32]:
for doc_id, text in documents.items():
    print("\n", doc_id)
    print(text)


 doc1
Students must submit assignments before the deadline.
Late submissions require instructor approval.
Students can request an extension when illness affects their work.
The request should include supporting medical evidence.
Assignments must be uploaded through the LMS.
Students should check the uploaded file before submission.
The instructor may reject incomplete submissions.
Students are responsible for keeping a copy of their work.
Students should read the assignment instructions carefully.
The deadline is mentioned on the learning portal.
Students must ensure that their files are readable.
If technical problems occur, students should contact IT support.
An extension should be requested before the deadline.
However, an extension is not automatically granted.
The instructor reviews each request individually.
Students should provide accurate information.
Late work may receive a penalty.
The penalty depends on course policy.
Students should avoid waiting until the last minute.
The

In [33]:
chunks = make_chunks(documents, "sentence")

for c in chunks:
    if c["doc"] == "doc1":
        print("\nChunk:", c["id"])
        print(c["text"])


Chunk: doc1:0
Students must submit assignments before the deadline.
Late submissions require instructor approval.
Students can request an extension when illness affects their work.
The request should include supporting medical evidence.
Assignments must be uploaded through the LMS.

Chunk: doc1:1

Students should check the uploaded file before submission.
The instructor may reject incomplete submissions.
Students are responsible for keeping a copy of their work.
Students should read the assignment instructions carefully.
The deadline is mentioned on the learning portal.

Chunk: doc1:2

Students must ensure that their files are readable.
If technical problems occur, students should contact IT support.
An extension should be requested before the deadline.
However, an extension is not automatically granted.
The instructor reviews each request individually.

Chunk: doc1:3

Students should provide accurate information.
Late work may receive a penalty.
The penalty depends on course policy.


In [34]:
chunks = make_chunks(documents, "sentence")

for c in chunks:
    if c["id"] == "doc1:0":
        print("Rule found:", "Late submissions require instructor approval." in c["text"])
        print("Exception found:", "Students can request an extension when illness affects their work." in c["text"])

Rule found: True
Exception found: True


In [35]:
chunks = make_chunks(documents, "10")

for doc_id in documents:
    doc_chunks = [c for c in chunks if c["doc"] == doc_id]

    print("\nDocument:", doc_id)

    for i in range(len(doc_chunks) - 1):
        first = doc_chunks[i]["text"]
        second = doc_chunks[i + 1]["text"]

        print(
            doc_chunks[i]["id"],
            "and",
            doc_chunks[i + 1]["id"],
            "Overlap words:",
            len(set(first.split()) & set(second.split()))
        )


Document: doc1
doc1:0 and doc1:1 Overlap words: 14
doc1:1 and doc1:2 Overlap words: 15
doc1:2 and doc1:3 Overlap words: 14
doc1:3 and doc1:4 Overlap words: 13

Document: doc2
doc2:0 and doc2:1 Overlap words: 17
doc2:1 and doc2:2 Overlap words: 17
doc2:2 and doc2:3 Overlap words: 14
doc2:3 and doc2:4 Overlap words: 14
doc2:4 and doc2:5 Overlap words: 10

Document: doc3
doc3:0 and doc3:1 Overlap words: 15
doc3:1 and doc3:2 Overlap words: 14
doc3:2 and doc3:3 Overlap words: 13
doc3:3 and doc3:4 Overlap words: 12
doc3:4 and doc3:5 Overlap words: 12


In [36]:
chunks = make_chunks(documents, "10")

for doc_id in documents:
    doc_chunks = [c for c in chunks if c["doc"] == doc_id]

    print("\nDocument:", doc_id)

    for i in range(len(doc_chunks) - 1):
        first_words = doc_chunks[i]["text"].split()
        second_words = doc_chunks[i + 1]["text"].split()

        print(
            doc_chunks[i]["id"],
            "and",
            doc_chunks[i + 1]["id"],
            "Exact overlap:",
            first_words[-10:] == second_words[:10]
        )


Document: doc1
doc1:0 and doc1:1 Exact overlap: True
doc1:1 and doc1:2 Exact overlap: True
doc1:2 and doc1:3 Exact overlap: True
doc1:3 and doc1:4 Exact overlap: True

Document: doc2
doc2:0 and doc2:1 Exact overlap: True
doc2:1 and doc2:2 Exact overlap: True
doc2:2 and doc2:3 Exact overlap: True
doc2:3 and doc2:4 Exact overlap: True
doc2:4 and doc2:5 Exact overlap: True

Document: doc3
doc3:0 and doc3:1 Exact overlap: True
doc3:1 and doc3:2 Exact overlap: True
doc3:2 and doc3:3 Exact overlap: True
doc3:3 and doc3:4 Exact overlap: True
doc3:4 and doc3:5 Exact overlap: True


In [37]:
print("Empty:", sentence_chunks(""))

print("Short:", sentence_chunks("Students should follow university rules."))

try:
    windows("This is a test.", 40, 40)
except ValueError as e:
    print("Invalid overlap:", e)

long_sentence = " ".join(["Students"] * 50) + "."
print("Long sentence:", sentence_chunks(long_sentence))

Empty: []
Short: [(0, 40)]
Invalid overlap: Invalid size or overlap
Long sentence: [(0, 359), (360, 450)]


In [38]:
text = "Dr. Rasikh teaches AI. Students attend his class."

print(sentence_chunks(text))

[(0, 49)]


In [39]:
documents["doc4"] = """
Library books must be returned by the due date.
Students may renew a book before the due date.
Renewal is not allowed if another student has reserved the book.
Students should keep their library cards safe.
Lost books must be reported to the librarian.
The library may charge a replacement fee.
Students should maintain silence in the reading area.
Food and drinks are not allowed near library books.
Students can use computers for academic work.
The librarian can help students find resources.
"""

documents["doc5"] = """
Students must pay their semester fees before registration.
The finance office provides payment instructions.
Students should keep their payment receipts.
Late fee may apply after the announced deadline.
Students can request a fee installment plan.
Installment approval depends on university policy.
Students should contact the finance office for questions.
Scholarship students must follow scholarship requirements.
Fee status can be checked on the student portal.
Students should verify payment before course registration.
"""

print("Total documents:", len(documents))

Total documents: 5


In [40]:
queries = [
    {
        "query": "When must assignments be submitted?",
        "answerable": True,
        "expected_doc": "doc1"
    },
    {
        "query": "Can illness allow an assignment extension?",
        "answerable": True,
        "expected_doc": "doc1"
    },
    {
        "query": "How do students connect to campus Wi-Fi?",
        "answerable": True,
        "expected_doc": "doc2"
    },
    {
        "query": "What should students do if Wi-Fi fails?",
        "answerable": True,
        "expected_doc": "doc2"
    },
    {
        "query": "How can students renew library books?",
        "answerable": True,
        "expected_doc": "doc4"
    },
    {
        "query": "What happens if a book is lost?",
        "answerable": True,
        "expected_doc": "doc4"
    },
    {
        "query": "Can students request fee installments?",
        "answerable": True,
        "expected_doc": "doc5"
    },
    {
        "query": "Where can students check their fee status?",
        "answerable": True,
        "expected_doc": "doc5"
    },
    {
        "query": "What is the university's cricket team selection policy?",
        "answerable": False,
        "expected_doc": None
    },
    {
        "query": "What are the cafeteria lunch prices?",
        "answerable": False,
        "expected_doc": None
    }
]

print("Total queries:", len(queries))

Total queries: 10


In [41]:
dev_queries = queries[:4]

test_queries = queries[4:8]

unanswerable_queries = queries[8:]

print("Development queries:", len(dev_queries))
print("Final test queries:", len(test_queries))
print("Unanswerable queries:", len(unanswerable_queries))

Development queries: 4
Final test queries: 4
Unanswerable queries: 2


In [42]:
for q in dev_queries:
    print(q)
    print()

{'query': 'When must assignments be submitted?', 'answerable': True, 'expected_doc': 'doc1'}

{'query': 'Can illness allow an assignment extension?', 'answerable': True, 'expected_doc': 'doc1'}

{'query': 'How do students connect to campus Wi-Fi?', 'answerable': True, 'expected_doc': 'doc2'}

{'query': 'What should students do if Wi-Fi fails?', 'answerable': True, 'expected_doc': 'doc2'}



In [45]:
chunks_0 = make_chunks(documents, "0")

print("Total chunks:", len(chunks_0))
print("First chunk:", chunks_0[0])

Total chunks: 19
First chunk: {'id': 'doc1:0', 'doc': 'doc1', 'start': 0, 'end': 294, 'text': 'Students must submit assignments before the deadline.\nLate submissions require instructor approval.\nStudents can request an extension when illness affects their work.\nThe request should include supporting medical evidence.\nAssignments must be uploaded through the LMS.\nStudents should check the'}


In [46]:
chunks_10 = make_chunks(documents, "10")

print("Total chunks:", len(chunks_10))
print("First chunk:", chunks_10[0])

Total chunks: 23
First chunk: {'id': 'doc1:0', 'doc': 'doc1', 'start': 0, 'end': 294, 'text': 'Students must submit assignments before the deadline.\nLate submissions require instructor approval.\nStudents can request an extension when illness affects their work.\nThe request should include supporting medical evidence.\nAssignments must be uploaded through the LMS.\nStudents should check the'}


In [47]:
chunks_sentence = make_chunks(documents, "sentence")

print("Total chunks:", len(chunks_sentence))
print("First chunk:", chunks_sentence[0])

Total chunks: 20
First chunk: {'id': 'doc1:0', 'doc': 'doc1', 'start': 0, 'end': 268, 'text': 'Students must submit assignments before the deadline.\nLate submissions require instructor approval.\nStudents can request an extension when illness affects their work.\nThe request should include supporting medical evidence.\nAssignments must be uploaded through the LMS.'}


In [48]:
print("Window 0:", len(chunks_0))
print("Window 10:", len(chunks_10))
print("Sentence-aware:", len(chunks_sentence))

Window 0: 19
Window 10: 23
Sentence-aware: 20


In [49]:
print("Window 0 chunk sizes:")
for chunk in chunks_0:
    print(chunk["id"], len(chunk["text"].split()))

print("\nWindow 10 chunk sizes:")
for chunk in chunks_10:
    print(chunk["id"], len(chunk["text"].split()))

print("\nSentence-aware chunk sizes:")
for chunk in chunks_sentence:
    print(chunk["id"], len(chunk["text"].split()))

Window 0 chunk sizes:
doc1:0 40
doc1:1 40
doc1:2 40
doc1:3 37
doc2:0 40
doc2:1 40
doc2:2 40
doc2:3 40
doc2:4 10
doc3:0 40
doc3:1 40
doc3:2 40
doc3:3 40
doc3:4 10
doc4:0 40
doc4:1 40
doc4:2 2
doc5:0 40
doc5:1 31

Window 10 chunk sizes:
doc1:0 40
doc1:1 40
doc1:2 40
doc1:3 40
doc1:4 37
doc2:0 40
doc2:1 40
doc2:2 40
doc2:3 40
doc2:4 40
doc2:5 20
doc3:0 40
doc3:1 40
doc3:2 40
doc3:3 40
doc3:4 40
doc3:5 20
doc4:0 40
doc4:1 40
doc4:2 22
doc5:0 40
doc5:1 40
doc5:2 11

Sentence-aware chunk sizes:
doc1:0 36
doc1:1 39
doc1:2 38
doc1:3 37
doc1:4 7
doc2:0 38
doc2:1 33
doc2:2 39
doc2:3 40
doc2:4 20
doc3:0 33
doc3:1 39
doc3:2 35
doc3:3 40
doc3:4 23
doc4:0 36
doc4:1 39
doc4:2 7
doc5:0 35
doc5:1 36


In [50]:
print("Window 0:")
for chunk in chunks_0[:3]:
    print(chunk["id"], chunk["start"], chunk["end"])

print("\nWindow 10:")
for chunk in chunks_10[:3]:
    print(chunk["id"], chunk["start"], chunk["end"])

print("\nSentence-aware:")
for chunk in chunks_sentence[:3]:
    print(chunk["id"], chunk["start"], chunk["end"])

Window 0:
doc1:0 0 294
doc1:1 295 578
doc1:2 579 875

Window 10:
doc1:0 0 294
doc1:1 235 512
doc1:2 437 717

Sentence-aware:
doc1:0 0 268
doc1:1 268 546
doc1:2 546 819


In [51]:
for chunk in chunks_0[:3]:
    original_text = documents[chunk["doc"]][chunk["start"]:chunk["end"]]

    print(chunk["id"], original_text == chunk["text"])

doc1:0 True
doc1:1 True
doc1:2 True


In [52]:
for chunk in chunks_10[:3]:
    original_text = documents[chunk["doc"]][chunk["start"]:chunk["end"]]

    print(chunk["id"], original_text == chunk["text"])

doc1:0 True
doc1:1 True
doc1:2 True


In [53]:
for chunk in chunks_sentence[:3]:
    original_text = documents[chunk["doc"]][chunk["start"]:chunk["end"]]

    print(chunk["id"], original_text == chunk["text"])

doc1:0 True
doc1:1 True
doc1:2 True


In [54]:
print("Window 0:", max(len(c["text"].split()) for c in chunks_0))
print("Window 10:", max(len(c["text"].split()) for c in chunks_10))
print("Sentence-aware:", max(len(c["text"].split()) for c in chunks_sentence))

Window 0: 40
Window 10: 40
Sentence-aware: 40


In [55]:
for chunk in chunks_0:
    if chunk["doc"] == "doc1":
        if "extension" in chunk["text"].lower():
            print("Chunk ID:", chunk["id"])
            print(chunk["text"])
            print("-" * 50)

Chunk ID: doc1:0
Students must submit assignments before the deadline.
Late submissions require instructor approval.
Students can request an extension when illness affects their work.
The request should include supporting medical evidence.
Assignments must be uploaded through the LMS.
Students should check the
--------------------------------------------------
Chunk ID: doc1:2
files are readable.
If technical problems occur, students should contact IT support.
An extension should be requested before the deadline.
However, an extension is not automatically granted.
The instructor reviews each request individually.
Students should provide accurate information.
Late work
--------------------------------------------------


In [56]:
for chunk in chunks_10:
    if chunk["doc"] == "doc1":
        if "extension" in chunk["text"].lower():
            print("Chunk ID:", chunk["id"])
            print(chunk["text"])
            print("-" * 50)

Chunk ID: doc1:0
Students must submit assignments before the deadline.
Late submissions require instructor approval.
Students can request an extension when illness affects their work.
The request should include supporting medical evidence.
Assignments must be uploaded through the LMS.
Students should check the
--------------------------------------------------
Chunk ID: doc1:2
Students should read the assignment instructions carefully.
The deadline is mentioned on the learning portal.
Students must ensure that their files are readable.
If technical problems occur, students should contact IT support.
An extension should be requested before the deadline.
--------------------------------------------------
Chunk ID: doc1:3
IT support.
An extension should be requested before the deadline.
However, an extension is not automatically granted.
The instructor reviews each request individually.
Students should provide accurate information.
Late work may receive a penalty.
The penalty depends on c

In [57]:
for chunk in chunks_sentence:
    if chunk["doc"] == "doc1":
        if "extension" in chunk["text"].lower():
            print("Chunk ID:", chunk["id"])
            print(chunk["text"])
            print("-" * 50)

Chunk ID: doc1:0
Students must submit assignments before the deadline.
Late submissions require instructor approval.
Students can request an extension when illness affects their work.
The request should include supporting medical evidence.
Assignments must be uploaded through the LMS.
--------------------------------------------------
Chunk ID: doc1:2

Students must ensure that their files are readable.
If technical problems occur, students should contact IT support.
An extension should be requested before the deadline.
However, an extension is not automatically granted.
The instructor reviews each request individually.
--------------------------------------------------


In [58]:
print("Empty:", sentence_chunks(""))
print("Short:", sentence_chunks("Students must submit assignments on time."))

Empty: []
Short: [(0, 41)]


In [59]:
long_text = " ".join(["Students must follow university rules"] * 10)

result = sentence_chunks(long_text)

print("Chunks:", result)

for start, end in result:
    print("Words:", len(long_text[start:end].split()))

Chunks: [(0, 303), (304, 379)]
Words: 40
Words: 10


In [60]:
try:
    windows("This is a test.", 40, 40)
except ValueError as e:
    print("Error handled:", e)

Error handled: Invalid size or overlap


In [61]:
text = "Dr. Rasikh teaches AI. Students attend his class."

result = sentence_chunks(text)

for start, end in result:
    print(text[start:end])
    print("-" * 30)

Dr. Rasikh teaches AI. Students attend his class.
------------------------------


In [62]:
text = "Dr. Rasikh teaches AI. " + " ".join(
    ["Students attend his class."] * 10
)

result = sentence_chunks(text)

for start, end in result:
    print(text[start:end])
    print("Words:", len(text[start:end].split()))
    print("-" * 30)

Dr. Rasikh teaches AI. Students attend his class. Students attend his class. Students attend his class. Students attend his class. Students attend his class. Students attend his class. Students attend his class. Students attend his class. Students attend his class. 
Words: 40
------------------------------
Students attend his class.
Words: 4
------------------------------


In [63]:
text = "Dr. Rasikh teaches AI. Students attend his class."

result = sentence_chunks(text)

for a, b in result:
    print(text[a:b])
    print("------------------------------")

Dr. Rasikh teaches AI. Students attend his class.
------------------------------


In [64]:
!pip install sentence-transformers

In [65]:
import sentence_transformers
print(sentence_transformers.__version__)

5.7.0


In [66]:
import time
from sentence_transformers import SentenceTransformer

start = time.perf_counter()

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu"
)

print("Model loaded successfully")
print("Load time:", round(time.perf_counter() - start, 2), "seconds")
print("Maximum sequence length:", model.max_seq_length)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully
Load time: 2.43 seconds
Maximum sequence length: 256


In [67]:
sample_text = chunks_0[0]["text"]

token_ids = model.tokenizer(
    sample_text,
    truncation=False
)["input_ids"]

print("Token count:", len(token_ids))
print("Model limit:", model.max_seq_length)
print("Fits within limit:", len(token_ids) <= model.max_seq_length)

Token count: 48
Model limit: 256
Fits within limit: True


In [68]:
all_chunks = chunks_0 + chunks_10 + chunks_sentence

token_counts = []

for chunk in all_chunks:
    ids = model.tokenizer(
        chunk["text"],
        truncation=False
    )["input_ids"]

    token_counts.append(len(ids))

print("Total chunks:", len(all_chunks))
print("Maximum token count:", max(token_counts))
print("Chunks within limit:", all(count <= model.max_seq_length for count in token_counts))
print("Chunks exceeding limit:", sum(count > model.max_seq_length for count in token_counts))

Total chunks: 62
Maximum token count: 52
Chunks within limit: True
Chunks exceeding limit: 0


In [69]:
texts = [chunk["text"] for chunk in all_chunks]

embeddings = model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embeddings shape:", embeddings.shape)
print("Total embeddings:", len(embeddings))

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embeddings shape: (62, 384)
Total embeddings: 62


In [70]:
import numpy as np

test_queries = [
    "What happens if a student submits an assignment late?",
    "How can students connect to the campus internet?",
    "What should group members do if someone does not complete their work?"
]

for query in test_queries:
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    scores = embeddings @ query_embedding
    top_indices = np.argsort(scores)[::-1][:3]

    print("\nQuery:", query)

    for index in top_indices:
        chunk = all_chunks[index]
        print("\nChunk ID:", chunk["id"])
        print("Score:", round(float(scores[index]), 3))
        print("Evidence:", chunk["text"])


Query: What happens if a student submits an assignment late?

Chunk ID: doc1:3
Score: 0.677
Evidence: 
Students should provide accurate information.
Late work may receive a penalty.
The penalty depends on course policy.
Students should avoid waiting until the last minute.
They should confirm successful submission.
The LMS may display a submission receipt.

Chunk ID: doc1:1
Score: 0.673
Evidence: 
Students should check the uploaded file before submission.
The instructor may reject incomplete submissions.
Students are responsible for keeping a copy of their work.
Students should read the assignment instructions carefully.
The deadline is mentioned on the learning portal.

Chunk ID: doc1:1
Score: 0.66
Evidence: uploaded file before submission.
The instructor may reject incomplete submissions.
Students are responsible for keeping a copy of their work.
Students should read the assignment instructions carefully.
The deadline is mentioned on the learning portal.
Students must ensure that the

In [71]:
all_chunks = []

for strategy, chunk_list in [
    ("window0", chunks_0),
    ("window10", chunks_10),
    ("sentence", chunks_sentence)
]:
    for chunk in chunk_list:
        new_chunk = chunk.copy()
        new_chunk["id"] = strategy + ":" + chunk["id"]
        new_chunk["strategy"] = strategy
        all_chunks.append(new_chunk)

texts = [chunk["text"] for chunk in all_chunks]

embeddings = model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Total chunks:", len(all_chunks))
print("Unique IDs:", len(set(chunk["id"] for chunk in all_chunks)))
print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Total chunks: 62
Unique IDs: 62
Embedding shape: (62, 384)


In [72]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(texts)

comparison_queries = [
    "What happens if a student submits an assignment late?",
    "How can students connect to the campus internet?",
    "What should group members do if someone does not complete their work?"
]

for query in comparison_queries:
    print("\nQUERY:", query)

    # TF-IDF search
    query_vector = vectorizer.transform([query])
    tfidf_scores = cosine_similarity(query_vector, tfidf_matrix)[0]
    tfidf_top = np.argsort(tfidf_scores)[::-1][:3]

    print("\nTF-IDF TOP RESULTS:")
    for index in tfidf_top:
        print(all_chunks[index]["id"], round(float(tfidf_scores[index]), 3))

    # Embedding search
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    embedding_scores = embeddings @ query_embedding
    embedding_top = np.argsort(embedding_scores)[::-1][:3]

    print("\nEMBEDDING TOP RESULTS:")
    for index in embedding_top:
        print(all_chunks[index]["id"], round(float(embedding_scores[index]), 3))


QUERY: What happens if a student submits an assignment late?

TF-IDF TOP RESULTS:
window0:doc1:2 0.28
window10:doc1:3 0.233
sentence:doc1:2 0.215

EMBEDDING TOP RESULTS:
sentence:doc1:3 0.677
sentence:doc1:1 0.673
window0:doc1:1 0.66

QUERY: How can students connect to the campus internet?

TF-IDF TOP RESULTS:
window0:doc2:1 0.249
window10:doc2:1 0.247
sentence:doc2:1 0.246

EMBEDDING TOP RESULTS:
sentence:doc2:2 0.781
window10:doc2:2 0.775
window10:doc2:0 0.729

QUERY: What should group members do if someone does not complete their work?

TF-IDF TOP RESULTS:
window10:doc3:4 0.451
window0:doc3:3 0.451
window10:doc3:0 0.437

EMBEDDING TOP RESULTS:
window10:doc3:5 0.616
window10:doc3:1 0.613
sentence:doc3:1 0.608


In [73]:
for query in comparison_queries:
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    scores = embeddings @ query_embedding
    top_indices = np.argsort(scores)[::-1][:3]

    print("\nQUERY:", query)

    for index in top_indices:
        chunk = all_chunks[index]

        print("\nID:", chunk["id"])
        print("Strategy:", chunk["strategy"])
        print("Score:", round(float(scores[index]), 3))
        print("Evidence:", chunk["text"])


QUERY: What happens if a student submits an assignment late?

ID: sentence:doc1:3
Strategy: sentence
Score: 0.677
Evidence: 
Students should provide accurate information.
Late work may receive a penalty.
The penalty depends on course policy.
Students should avoid waiting until the last minute.
They should confirm successful submission.
The LMS may display a submission receipt.

ID: sentence:doc1:1
Strategy: sentence
Score: 0.673
Evidence: 
Students should check the uploaded file before submission.
The instructor may reject incomplete submissions.
Students are responsible for keeping a copy of their work.
Students should read the assignment instructions carefully.
The deadline is mentioned on the learning portal.

ID: window0:doc1:1
Strategy: window0
Score: 0.66
Evidence: uploaded file before submission.
The instructor may reject incomplete submissions.
Students are responsible for keeping a copy of their work.
Students should read the assignment instructions carefully.
The deadline is

In [76]:
import json

documents["doc6"] = """
An older library notice states that students may renew books
after the due date. This notice conflicts with the current
library rule. Students should follow the current library
policy and confirm uncertain cases with the librarian.
"""

with open("corpus.json", "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=4)

print("Corpus saved:", len(documents), "documents")

Corpus saved: 6 documents


In [77]:
%%writefile task3_retrieval.py

import json
import csv
import time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from task2_chunks import make_chunks

with open("corpus.json", encoding="utf-8") as f:
    documents = json.load(f)

queries = [
    {"query": "When must assignments be submitted?", "doc": "doc1",
     "evidence": "Students must submit assignments before the deadline.", "split": "dev"},
    {"query": "Can illness allow an assignment extension?", "doc": "doc1",
     "evidence": "Students can request an extension when illness affects their work.", "split": "dev"},
    {"query": "How do students connect to campus Wi-Fi?", "doc": "doc2",
     "evidence": "Students must use their university login credentials.", "split": "dev"},
    {"query": "What should students do if Wi-Fi fails?", "doc": "doc2",
     "evidence": "If the connection fails, students should contact IT support.", "split": "dev"},
    {"query": "How can students renew library books?", "doc": "doc4",
     "evidence": "Students may renew a book before the due date.", "split": "test"},
    {"query": "What happens if a book is lost?", "doc": "doc4",
     "evidence": "Lost books must be reported to the librarian.", "split": "test"},
    {"query": "Can students request fee installments?", "doc": "doc5",
     "evidence": "Students can request a fee installment plan.", "split": "test"},
    {"query": "Where can students check their fee status?", "doc": "doc5",
     "evidence": "Fee status can be checked on the student portal.", "split": "test"},
    {"query": "What is the university cricket team selection policy?", "doc": None,
     "evidence": None, "split": "test"},
    {"query": "What are the cafeteria lunch prices?", "doc": None,
     "evidence": None, "split": "test"}
]

def evaluate(scores, chunks, items):
    hits = []
    reciprocal = []

    for row, q in zip(scores, items):
        if q["doc"] is None:
            continue

        top = np.argsort(-row, kind="stable")[:3]
        ranks = [
            rank for rank, i in enumerate(top, 1)
            if chunks[i]["doc"] == q["doc"]
            and q["evidence"] in chunks[i]["text"]
        ]

        hits.append(bool(ranks))
        reciprocal.append(1 / ranks[0] if ranks else 0)

    return round(float(np.mean(hits)), 3), round(float(np.mean(reciprocal)), 3)

start = time.perf_counter()
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu"
)
print("Model load time:", round(time.perf_counter() - start, 3), "seconds")

logs = []
results = []

for strategy in ["0", "10", "sentence"]:
    chunks = make_chunks(documents, strategy)
    texts = [c["text"] for c in chunks]
    query_texts = [q["query"] for q in queries]

    lengths = [
        len(model.tokenizer(t, truncation=False)["input_ids"])
        for t in texts + query_texts
    ]

    print("\nStrategy:", strategy)
    print("Chunk count:", len(chunks))
    print("Maximum token length:", max(lengths))
    print("Within model limit:", max(lengths) <= model.max_seq_length)

    if max(lengths) > model.max_seq_length:
        raise ValueError("Model token limit exceeded")

    for method in ["tfidf", "embedding"]:
        t = time.perf_counter()

        if method == "tfidf":
            vectorizer = TfidfVectorizer()
            C = vectorizer.fit_transform(texts)
            Q = vectorizer.transform(query_texts)
        else:
            C = model.encode(texts, normalize_embeddings=True)
            Q = model.encode(query_texts, normalize_embeddings=True)

        encoding_time = time.perf_counter() - t

        t = time.perf_counter()
        scores = cosine_similarity(Q, C) if method == "tfidf" else Q @ C.T
        search_time = time.perf_counter() - t

        dev_indices = [i for i, q in enumerate(queries) if q["split"] == "dev"]
        test_indices = [i for i, q in enumerate(queries) if q["split"] == "test"]

        dev = evaluate(scores[dev_indices], chunks,
                       [queries[i] for i in dev_indices])
        test = evaluate(scores[test_indices], chunks,
                        [queries[i] for i in test_indices])

        print("\nMethod:", method)
        print("Encoding time:", round(encoding_time, 4))
        print("Search time:", round(search_time, 4))
        print("Development Hit@3, MRR@3:", dev)
        print("Test Hit@3, MRR@3:", test)

        results.append({
            "strategy": strategy,
            "method": method,
            "dev_mrr": dev[1],
            "dev_hit": dev[0],
            "test": test
        })

        for row, q in zip(scores, queries):
            top = np.argsort(-row, kind="stable")[:3]

            for rank, i in enumerate(top, 1):
                c = chunks[i]
                record = {
                    "strategy": strategy,
                    "method": method,
                    "query": q["query"],
                    "split": q["split"],
                    "rank": rank,
                    "score": round(float(row[i]), 4),
                    "chunk_id": c["id"],
                    "doc": c["doc"],
                    "start": c["start"],
                    "end": c["end"],
                    "text": c["text"],
                    "answerable": q["doc"] is not None
                }
                logs.append(record)
                print("Top", rank, c["id"], record["score"],
                      c["start"], c["end"], c["text"][:150])

best = max(results, key=lambda x: (x["dev_mrr"], x["dev_hit"]))

print("\nSelected on development data:")
print(best["strategy"], best["method"])
print("Development:", best["dev_hit"], best["dev_mrr"])
print("Held-out test:", best["test"])

with open("retrieval_log.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=logs[0].keys())
    writer.writeheader()
    writer.writerows(logs)

print("\nCSV saved: retrieval_log.csv")

Overwriting task3_retrieval.py


In [78]:
!python task3_retrieval.py

Loading weights: 100% 103/103 [00:00<00:00, 3706.61it/s]
Model load time: 1.604 seconds
Traceback (most recent call last):
  File "/content/task3_retrieval.py", line 68, in <module>
    chunks = make_chunks(documents, strategy)
  File "/content/task2_chunks.py", line 8, in make_chunks
    spans = windows(text, 40, int(strategy))
            ^^^^^^^
NameError: name 'windows' is not defined


In [79]:
!cat task2_chunks.py

def make_chunks(docs, strategy):
    result = []

    for doc_id, text in docs.items():
        if strategy == "sentence":
            spans = sentence_chunks(text)
        else:
            spans = windows(text, 40, int(strategy))

        for i, (a, b) in enumerate(spans):
            result.append({
                "id": f"{doc_id}:{i}",
                "doc": doc_id,
                "start": a,
                "end": b,
                "text": text[a:b]
            })

    return result


In [80]:
%%writefile task2_chunks.py

def windows(text, size=40, overlap=0):
    words = text.split()
    spans = []
    step = size - overlap

    for i in range(0, len(words), step):
        start = len(" ".join(words[:i]))
        end = start + len(" ".join(words[i:i + size]))
        spans.append((start, end))

    return spans


def sentence_chunks(text):
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    spans = []
    start = 0

    for sentence in sentences:
        if sentence:
            end = start + len(sentence)
            spans.append((start, end))
            start = end + 1

    return spans


def make_chunks(docs, strategy):
    result = []

    for doc_id, text in docs.items():
        if strategy == "sentence":
            spans = sentence_chunks(text)
        else:
            spans = windows(text, 40, int(strategy))

        for i, (a, b) in enumerate(spans):
            result.append({
                "id": f"{strategy}_{doc_id}:{i}",
                "doc": doc_id,
                "start": a,
                "end": b,
                "text": text[a:b]
            })

    return result

Overwriting task2_chunks.py


In [81]:
!python task3_retrieval.py

Loading weights: 100% 103/103 [00:00<00:00, 4500.80it/s]
Model load time: 1.849 seconds

Strategy: 0
Chunk count: 20
Maximum token length: 52
Within model limit: True

Method: tfidf
Encoding time: 0.0051
Search time: 0.0011
Development Hit@3, MRR@3: (1.0, 0.875)
Test Hit@3, MRR@3: (0.75, 0.75)
Top 1 0_doc1:0 0.4569 0 294 Students must submit assignments before the deadline.
Late submissions require instructor approval.
Students can request an extension when illness aff
Top 2 0_doc2:0 0.1283 0 278 Campus Wi-Fi is available to registered students.
Students must use their university login credentials.
The network should only be used for educationa
Top 3 0_doc4:0 0.1184 0 225 
Library books must be returned by the due date.
Students may renew a book before the due date.
Renewal is not allowed if another student has reserved
Top 1 0_doc1:2 0.2891 578 874  files are readable.
If technical problems occur, students should contact IT support.
An extension should be requested before the deadline

In [82]:
import os
print(os.path.exists("retrieval_log.csv"))

True


In [83]:
%%writefile document_chatbot.py

import json
import numpy as np
from sentence_transformers import SentenceTransformer
from task2_chunks import make_chunks

with open("corpus.json", encoding="utf-8") as f:
    documents = json.load(f)

chunks = make_chunks(documents, "0")

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

texts = [c["text"] for c in chunks]

embeddings = model.encode(
    texts, normalize_embeddings=True
)

def answer_query(query):
    if not query.strip():
        print("Please enter a question.")
        return

    q_embedding = model.encode(
        [query], normalize_embeddings=True
    )

    scores = q_embedding @ embeddings.T
    top = np.argsort(-scores[0])[:3]

    print("\nPossible evidence:")

    for rank, i in enumerate(top, 1):
        c = chunks[i]
        print("\n", rank, "Score:", round(float(scores[0][i]), 3))
        print("Document:", c["doc"])
        print("Offset:", c["start"], "-", c["end"])
        print(c["text"])

    choice = input(
        "\nDo you want to view the retrieved evidence? (yes/no): "
    )

    if choice.lower() == "yes":
        print("\nEvidence displayed above.")
    else:
        print("No evidence confirmed. Please rephrase your question.")

print("Document Chatbot started. Type exit to stop.")

while True:
    query = input("\nYou: ")

    if query.strip().lower() == "exit":
        break

    answer_query(query)

Writing document_chatbot.py


In [85]:
!python document_chatbot.py

Loading weights: 100% 103/103 [00:00<00:00, 3649.32it/s]
Document Chatbot started. Type exit to stop.

You: When must assignments be submitted?

Possible evidence:

 1 Score: 0.716
Document: doc1
Offset: 0 - 294
Students must submit assignments before the deadline.
Late submissions require instructor approval.
Students can request an extension when illness affects their work.
The request should include supporting medical evidence.
Assignments must be uploaded through the LMS.
Students should check the

 2 Score: 0.7
Document: doc3
Offset: 1142 - 1214
 assignment.
The group should confirm the submission before the deadline

 3 Score: 0.668
Document: doc1
Offset: 294 - 577
 uploaded file before submission.
The instructor may reject incomplete submissions.
Students are responsible for keeping a copy of their work.
Students should read the assignment instructions carefully.
The deadline is mentioned on the learning portal.
Students must ensure that thei

Do you want to view the retrieved e

In [86]:
%%writefile document_chatbot.py

import json
import numpy as np
from sentence_transformers import SentenceTransformer
from task2_chunks import make_chunks

with open("corpus.json", encoding="utf-8") as f:
    documents = json.load(f)

chunks = make_chunks(documents, "0")

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

texts = [c["text"] for c in chunks]
embeddings = model.encode(texts, normalize_embeddings=True)

# Threshold calibrated using known answerable and unanswerable queries
dev_queries = [
    ("When must assignments be submitted?", 1),
    ("How can students renew library books?", 1),
    ("How do students check fee status?", 1),
    ("What are cafeteria lunch prices?", 0),
]

dev_scores = []

for query, label in dev_queries:
    q = model.encode([query], normalize_embeddings=True)
    score = float((q @ embeddings.T).max())
    dev_scores.append((score, label))

thresholds = sorted(set(score for score, label in dev_scores))

best_threshold = thresholds[0]
best_accuracy = -1

for threshold in thresholds:
    correct = sum(
        (score >= threshold) == bool(label)
        for score, label in dev_scores
    )
    accuracy = correct / len(dev_scores)

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_threshold = threshold

print("Calibrated threshold:", round(best_threshold, 3))

def answer_query(query):
    if not query.strip():
        print("Please enter a question.")
        return

    q = model.encode([query], normalize_embeddings=True)
    scores = (q @ embeddings.T)[0]
    top = np.argsort(-scores)[:3]

    if float(scores[top[0]]) < best_threshold:
        print("No reliable evidence found in the documents.")
        print("Please check the official source.")
        return

    print("\nRetrieved evidence candidates:")

    for rank, i in enumerate(top, 1):
        c = chunks[i]
        print(
            rank, "| Score:", round(float(scores[i]), 3),
            "| Document:", c["doc"],
            "| Offset:", c["start"], "-", c["end"]
        )

    choice = input("\nView evidence? (yes/no): ").strip().lower()

    if choice == "yes":
        for i in top:
            c = chunks[i]
            print("\n", c["doc"], "|", c["start"], "-", c["end"])
            print(c["text"])
    else:
        print("Evidence not confirmed. Please rephrase your question.")

print("\nDocument Chatbot started. Type exit to stop.")

while True:
    query = input("\nYou: ")

    if query.strip().lower() == "exit":
        break

    answer_query(query)

Overwriting document_chatbot.py


In [87]:
!python document_chatbot.py

Loading weights: 100% 103/103 [00:00<00:00, 12393.52it/s]
Calibrated threshold: 0.653

Document Chatbot started. Type exit to stop.

You: When must assignments be submitted?

Retrieved evidence candidates:
1 | Score: 0.716 | Document: doc1 | Offset: 0 - 294
2 | Score: 0.7 | Document: doc3 | Offset: 1142 - 1214
3 | Score: 0.668 | Document: doc1 | Offset: 294 - 577

View evidence? (yes/no): no
Evidence not confirmed. Please rephrase your question.

You: What are the cafeteria lunch prices?
No reliable evidence found in the documents.
Please check the official source.

You: How can students renew library books?

Retrieved evidence candidates:
1 | Score: 0.686 | Document: doc4 | Offset: 0 - 225
2 | Score: 0.686 | Document: doc6 | Offset: 0 - 231
3 | Score: 0.622 | Document: doc4 | Offset: 225 - 477

View evidence? (yes/no): yes

 doc4 | 0 - 225

Library books must be returned by the due date.
Students may renew a book before the due date.
Renewal is not allowed if another student has reser